# 00: Настройка окружения
Монтирование диска, установка пакетов, инициализация конфига и логгера.

In [1]:
import sys
from pathlib import Path

# Определяем корневые пути локально (папка проекта quntum-classic)
PROJECT_ROOT = Path.cwd().parent  # если ноутбук в notebooks/, то корень - на уровень выше
SRC_PATH = PROJECT_ROOT / "src"
CONFIG_PATH = PROJECT_ROOT / "configs"
OUTPUT_PATH = PROJECT_ROOT / "outputs"

# Создаём папки, если их нет
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)
(OUTPUT_PATH / "models").mkdir(exist_ok=True)
(OUTPUT_PATH / "logs").mkdir(exist_ok=True)

# Добавляем src в PYTHONPATH
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

print(f"Project root: {PROJECT_ROOT}")
print(f"Python path includes src: {str(SRC_PATH) in sys.path}")

Project root: c:\Projects\quntum-classic
Python path includes src: True


In [2]:
import subprocess
import sys
from pathlib import Path

def install_requirements(requirements_path: Path):
    """Установка пакетов из requirements.txt."""
    if not requirements_path.exists():
        print(f"File {requirements_path} not found, skipping installation.")
        return
    
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-r", str(requirements_path), "--quiet"],
        capture_output=True,
        text=True
    )
    
    if result.returncode != 0:
        print(f"Error installing packages:\n{result.stderr}")
    else:
        print("All packages installed successfully.")

# Установка пакетов (закомментируй, если не нужно)
# install_requirements(PROJECT_ROOT / "requirements.txt")
print("Local mode: packages are managed by uv. Skipping Colab install.")

Local mode: packages are managed by uv. Skipping Colab install.


In [4]:
from core.config_loader import init_config, get_config
from core.logger import setup_logging

# Инициализируем конфиг (локальный путь)
init_config(CONFIG_PATH / "config.yaml")
cfg = get_config()

# Настраиваем логирование
setup_logging(
    log_level=cfg.logging.level,
    log_format=cfg.logging.format,
    log_file=OUTPUT_PATH / cfg.logging.file
)

print(f"Environment: {cfg.environment}")
print(f"Backend (QCloud chip): {cfg.backends.qcloud.chip}")
print(f"Backend (Octillion chip): {cfg.backends.octillion.chip}")
print(f"Logging level: {cfg.logging.level}")
print("Configuration and logging initialized.")

2026-09-05 19:16:07,532 - root - INFO - Logging initialized. Level: INFO
Environment: colab
Backend (QCloud chip): WK_C180
Backend (Octillion chip): Snowdrop 8q ver2
Logging level: INFO
Configuration and logging initialized.


In [5]:
import os

def load_secrets():
    """Загружает API-ключи из переменных окружения Windows."""
    secrets = {
        'QPANDA_QCLOUD_API_KEY': 'QCLOUD_API_KEY',
        'OCTILLION_TOKEN': 'OCTILLION_TOKEN',
        'WANDB_API_KEY': 'WANDB_API_KEY',
    }
    
    for env_var, secret_name in secrets.items():
        value = os.environ.get(env_var)
        if value:
            print(f"✓ {env_var} loaded from environment")
        else:
            print(f"⚠ {env_var} not set. Add it via: setx {env_var} YOUR_VALUE")

load_secrets()

⚠ QPANDA_QCLOUD_API_KEY not set. Add it via: setx QPANDA_QCLOUD_API_KEY YOUR_VALUE
⚠ OCTILLION_TOKEN not set. Add it via: setx OCTILLION_TOKEN YOUR_VALUE
⚠ WANDB_API_KEY not set. Add it via: setx WANDB_API_KEY YOUR_VALUE


In [6]:
import sys

def reload_project_modules():
    """Перезагружает все модули проекта после редактирования .py файлов."""
    modules_order = ['core', 'utils', 'pipelines']
    
    for prefix in modules_order:
        to_reload = [
            name for name in sys.modules
            if name.startswith(prefix) or name.startswith(f"src.{prefix}")
        ]
        for name in sorted(to_reload, reverse=True):
            if name in sys.modules:
                del sys.modules[name]
    
    print("Project modules cache cleared. Ready for fresh imports.")

reload_project_modules()

Project modules cache cleared. Ready for fresh imports.
